# 2.13 Integer & combinatorial optimization

Discrete optimization is hard because the average of two valid choices is often not a valid choice.

LP relaxations provide bounds by allowing fractional choices, while duality explains why bounds can prune branches before an integer solution is found. Feature selection, routing, assignment, model compression, and scheduling all contain choices that cannot be averaged away.

Save a copy to Drive to edit.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt

SEED = 213
rng = np.random.default_rng(SEED)


def sigmoid(scores):
    return 1.0 / (1.0 + np.exp(-np.clip(scores, -30, 30)))


def load_small_binary_data(max_features=12):
    try:
        from sklearn.datasets import load_breast_cancer

        data = load_breast_cancer()
        features = data.data[:, :max_features].astype(float)
        target = data.target.astype(float)
    except Exception:
        features = rng.normal(size=(180, max_features))
        true_weights = np.linspace(-1.2, 1.2, max_features)
        logits = features @ true_weights
        target = (sigmoid(logits) > 0.5).astype(float)

    split = int(0.7 * len(features))
    train_x = features[:split]
    valid_x = features[split:]
    train_y = target[:split]
    valid_y = target[split:]
    mean = train_x.mean(axis=0)
    scale = train_x.std(axis=0) + 1e-6
    train_x = (train_x - mean) / scale
    valid_x = (valid_x - mean) / scale
    return train_x, train_y, valid_x, valid_y


def logistic_validation_loss(train_x, train_y, valid_x, valid_y, feature_mask, steps=80, learning_rate=0.15, l2=1e-3):
    selected = np.where(feature_mask == 1)[0]
    if len(selected) == 0:
        baseline = np.full_like(valid_y, train_y.mean())
        return -float(np.mean(valid_y * np.log(baseline + 1e-9) + (1.0 - valid_y) * np.log(1.0 - baseline + 1e-9)))

    x_train = train_x[:, selected]
    x_valid = valid_x[:, selected]
    weights = np.zeros(x_train.shape[1])
    bias = 0.0

    for step in range(steps):
        prediction = sigmoid(x_train @ weights + bias)
        error = prediction - train_y
        gradient = x_train.T @ error / len(train_y) + l2 * weights
        bias_gradient = float(np.mean(error))
        weights = weights - learning_rate * gradient
        bias = bias - learning_rate * bias_gradient

    valid_prediction = sigmoid(x_valid @ weights + bias)
    loss = -np.mean(valid_y * np.log(valid_prediction + 1e-9) + (1.0 - valid_y) * np.log(1.0 - valid_prediction + 1e-9))
    return float(loss)


def logistic_feature_values(max_features=12):
    train_x, train_y, valid_x, valid_y = load_small_binary_data(max_features=max_features)
    empty_mask = np.zeros(max_features, dtype=int)
    baseline_loss = logistic_validation_loss(train_x, train_y, valid_x, valid_y, empty_mask)
    values = []

    for index in range(max_features):
        mask = np.zeros(max_features, dtype=int)
        mask[index] = 1
        one_feature_loss = logistic_validation_loss(train_x, train_y, valid_x, valid_y, mask)
        values.append(max(0.01, baseline_loss - one_feature_loss))

    scaled_values = 100.0 * np.asarray(values)
    weights = 1 + (np.arange(max_features) % 4)
    return weights, scaled_values


def fractional_upper_bound(weights, values, capacity, start_index, current_weight, current_value, order):
    remaining = capacity - current_weight
    bound = float(current_value)

    for position in range(start_index, len(order)):
        item = order[position]
        weight = float(weights[item])
        value = float(values[item])

        if weight <= remaining:
            remaining = remaining - weight
            bound = bound + value
        else:
            if weight > 0:
                bound = bound + value * remaining / weight
            break

    return bound


def knapsack_branch_and_bound(weights, values, capacity):
    weights = np.asarray(weights, dtype=float)
    values = np.asarray(values, dtype=float)
    n_items = len(weights)
    density = values / np.maximum(weights, 1e-12)
    order = np.argsort(-density)
    best_value = -np.inf
    best_mask = np.zeros(n_items, dtype=int)
    trace = []
    nodes_seen = 0

    def search(position, current_weight, current_value, mask):
        nonlocal best_value
        nonlocal best_mask
        nonlocal nodes_seen

        nodes_seen = nodes_seen + 1
        bound = fractional_upper_bound(weights, values, capacity, position, current_weight, current_value, order)
        trace.append((nodes_seen, float(current_value), float(bound), float(best_value)))

        if bound <= best_value + 1e-12:
            return

        if position == n_items:
            if current_value > best_value and current_weight <= capacity:
                best_value = float(current_value)
                best_mask = mask.copy()
            return

        item = order[position]
        next_weight = current_weight + weights[item]

        if next_weight <= capacity:
            mask[item] = 1
            search(position + 1, next_weight, current_value + values[item], mask)
            mask[item] = 0

        search(position + 1, current_weight, current_value, mask)

    search(0, 0.0, 0.0, np.zeros(n_items, dtype=int))

    return best_mask, best_value, np.asarray(trace, dtype=float)


def fractional_relaxation(weights, values, capacity):
    weights = np.asarray(weights, dtype=float)
    values = np.asarray(values, dtype=float)
    density = values / np.maximum(weights, 1e-12)
    order = np.argsort(-density)
    fractions = np.zeros(len(weights), dtype=float)
    remaining = float(capacity)

    for item in order:
        if weights[item] <= remaining:
            fractions[item] = 1.0
            remaining = remaining - weights[item]
        else:
            fractions[item] = remaining / weights[item]
            break

    return fractions


def repair_mask(mask, weights, values, capacity):
    mask = np.asarray(mask, dtype=int).copy()
    weights = np.asarray(weights, dtype=float)
    values = np.asarray(values, dtype=float)
    density = values / np.maximum(weights, 1e-12)

    while np.dot(mask, weights) > capacity:
        selected = np.where(mask == 1)[0]
        remove_index = selected[np.argmin(density[selected])]
        mask[remove_index] = 0

    return mask


def build_integer_ladder():
    d1 = {
        "name": "D1 lesson knapsack",
        "weights": np.array([2, 3, 4]),
        "values": np.array([3, 4, 5]),
        "capacity": 5,
    }
    d2 = {
        "name": "D2 ill-conditioned item scales",
        "weights": np.array([1, 2, 8, 9, 10, 20, 21, 22]),
        "values": np.array([1, 4, 13, 14, 15, 25, 27, 28]),
        "capacity": 31,
    }
    d3 = {
        "name": "D3 multimodal binary rewards",
        "weights": np.array([3, 4, 5, 6, 7, 8, 9, 10, 12, 13]),
        "values": np.array([8, 9, 10, 11, 13, 14, 15, 17, 18, 19]),
        "capacity": 30,
    }
    d4_weights, d4_values = logistic_feature_values(max_features=12)
    d4 = {
        "name": "D4 real logistic-regression feature subset",
        "weights": d4_weights,
        "values": d4_values,
        "capacity": 13,
    }
    d5 = {
        "name": "D5 constrained high-dimensional subset",
        "weights": np.array([2, 3, 4, 5, 8, 9, 7, 6, 10, 11, 13, 14, 1, 4, 6, 12, 15, 16]),
        "values": np.array([4, 5, 7, 9, 15, 16, 13, 10, 17, 18, 22, 23, 2, 8, 11, 20, 24, 25]),
        "capacity": 45,
    }

    return [d1, d2, d3, d4, d5]


## The concept, built once: branch-and-bound for 0-1 knapsack

A 0-1 integer program uses $z_j \in \{0,1\}$. For knapsack, maximize $v^Tz$ subject to $w^Tz \le C$.

The reusable solver below branches on include/exclude decisions and uses a fractional knapsack upper bound to prune nodes that cannot beat the incumbent.

In [ ]:

lesson_weights = np.array([2, 3, 4])
lesson_values = np.array([3, 4, 5])
lesson_capacity = 5

lesson_mask, lesson_value, lesson_trace = knapsack_branch_and_bound(lesson_weights, lesson_values, lesson_capacity)

print("best mask", lesson_mask.tolist())
print("best value", lesson_value)
print("nodes visited", len(lesson_trace))

assert lesson_mask.tolist() == [1, 1, 0]
assert lesson_value == 7
assert np.dot(lesson_mask, lesson_weights) == 5


## Relaxed bounds are not integer answers

The lesson's relaxation gap uses weights $(4,5)$, values $(7,10)$, and capacity $6$. The integer answer takes item 2 for value $10$, while a fractional fill takes $0.25$ of item 1 for an upper bound $10+0.25\cdot7=11.75$.

In [ ]:

gap_weights = np.array([4, 5])
gap_values = np.array([7, 10])
gap_capacity = 6

integer_mask, integer_value, gap_trace = knapsack_branch_and_bound(gap_weights, gap_values, gap_capacity)
fractional_choice = fractional_relaxation(gap_weights, gap_values, gap_capacity)
fractional_value = float(np.dot(fractional_choice, gap_values))
relaxation_gap = fractional_value - integer_value

print("integer", integer_mask.tolist(), integer_value)
print("fractional", fractional_choice.round(4).tolist(), fractional_value)
print("gap", relaxation_gap)

assert integer_mask.tolist() == [0, 1]
assert integer_value == 10
assert abs(fractional_choice[0] - 0.25) < 1e-12
assert abs(fractional_value - 11.75) < 1e-12
assert abs(relaxation_gap - 1.75) < 1e-12


## The dataset ladder

D1 is the exact lesson knapsack. D2 stretches item scales, D3 creates several nearly competing modes, D4 derives item gains from real NumPy logistic-regression validation losses, and D5 is a larger constrained subset where branch-and-bound certificates matter.

In [ ]:

rungs = build_integer_ladder()

for index, rung in enumerate(rungs, start=1):
    weights = rung["weights"]
    values = rung["values"]
    capacity = rung["capacity"]
    print(f"D{index}: {rung['name']}")
    print("  items", len(weights), "capacity", capacity)
    print("  first weights", weights[:6].tolist())
    print("  first values", values[:6].tolist())


## Run the same method across D1-D5

Every rung uses the same `knapsack_branch_and_bound()` routine. The metric is best feasible objective; larger is better.

In [ ]:

integer_results = []

for index, rung in enumerate(rungs, start=1):
    mask, best_value, trace = knapsack_branch_and_bound(rung["weights"], rung["values"], rung["capacity"])
    weight = float(np.dot(mask, rung["weights"]))
    integer_results.append({
        "rung": f"D{index}",
        "name": rung["name"],
        "mask": mask,
        "best": best_value,
        "weight": weight,
        "trace": trace,
    })
    print(f"D{index} | best objective {best_value:7.3f} | weight {weight:5.1f} | nodes {len(trace):4d}")


## Results visualization

The left panels show which items the search selects. The right panel shows each rung's incumbent objective over branch-and-bound node visits.

In [ ]:

fig, axes = plt.subplots(2, 3, figsize=(15, 7))
flat_axes = axes.ravel()

for index, result in enumerate(integer_results):
    ax = flat_axes[index]
    selected = result["mask"]
    ax.bar(np.arange(len(selected)), selected)
    ax.set_title(f"{result['rung']} selected items")
    ax.set_ylim(-0.05, 1.05)
    ax.set_xlabel("item")
    ax.set_ylabel("z")

summary_ax = flat_axes[-1]
for result in integer_results:
    trace = result["trace"]
    incumbent = np.maximum.accumulate(trace[:, 3])
    summary_ax.plot(np.arange(len(incumbent)), incumbent, label=result["rung"])

summary_ax.set_title("best objective vs iteration")
summary_ax.set_xlabel("branch-and-bound node")
summary_ax.set_ylabel("incumbent value")
summary_ax.legend()
plt.tight_layout()


## Pitfall on D5: rounding a fractional relaxation

Fractional LP variables satisfy relaxed constraints, not integer constraints. Rounding every positive fraction to $1$ can make $w^Tz \gt C$. The fix is to repair feasibility and report both the relaxed bound and the feasible incumbent.

In [ ]:

d5 = rungs[-1]
fractional = fractional_relaxation(d5["weights"], d5["values"], d5["capacity"])
naive = (fractional > 0).astype(int)
repaired = repair_mask(naive, d5["weights"], d5["values"], d5["capacity"])
exact_mask, exact_value, exact_trace = knapsack_branch_and_bound(d5["weights"], d5["values"], d5["capacity"])

fractional_bound = float(np.dot(fractional, d5["values"]))
naive_weight = float(np.dot(naive, d5["weights"]))
repaired_value = float(np.dot(repaired, d5["values"]))
repaired_weight = float(np.dot(repaired, d5["weights"]))

print("fractional upper bound", fractional_bound)
print("naive rounded weight", naive_weight, "capacity", d5["capacity"])
print("repaired value", repaired_value, "repaired weight", repaired_weight)
print("exact incumbent", exact_value)

assert naive_weight > d5["capacity"]
assert repaired_weight <= d5["capacity"]
assert exact_value >= repaired_value


## Evaluate it + Practice

- Metric: track best feasible objective against a no-skill baseline: random feasible masks under the same capacity.
- Cheap sanity check: D1 must return mask [1, 1, 0] with value 7.
- Ablation: replace the fractional bound with no bound and watch node visits increase.
- Failure signals: rounded masks are infeasible, incumbent stays flat, or bound is mistaken for a solution.

### Practice

- Change D3 capacity and predict which competing subset should win before running it.

- Add a cardinality cap to D5 and adapt the bound so pruning stays valid.

- Compare best density-greedy value with the branch-and-bound incumbent on each rung.